<a href="https://colab.research.google.com/github/anushabuilds/Textgrad_experiments/blob/main/TextGrad_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TextGrad Tutorial - Single and Multi Turn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook has some of my **TextGrad** experiment in multiple settings - an autograd engine for text optimization - using **OpenRouter** API.

The idea behind TextGrad is that it brings automatic differentiation to text. Think of it as being analogous to PyTorch's autograd for numerical optimization. TextGrad uses LLM feedback as gradients to optimize prompts, solutions to problems, code snippets or any text that affects output quality



In [48]:
# Install TextGrad:
!pip install textgrad -q
print(" TextGrad installed successfully!")

 TextGrad installed successfully!


In [60]:
import os
import textgrad as tg


os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Use Colab secrets to add your OpenRouter or other API key
print(" API key configured!")

 API key configured!


# Configuring TextGrad

Setting up the **backward engine** - the LLM that provides feedback/gradients.


In [61]:
# Set the backward engine (provides "gradients" / feedback)
# Format: experimental:openrouter/<provider>/<model>

BACKWARD_MODEL = "experimental:openrouter/anthropic/claude-3.5-sonnet"
tg.set_backward_engine(BACKWARD_MODEL, override=True, cache=True)

print("TextGrad configured!")
print(f"   Backward engine: {BACKWARD_MODEL}")

TextGrad configured!
   Backward engine: experimental:openrouter/anthropic/claude-3.5-sonnet


---

# Example 1: Single-Turn Solution Optimization

Fix an incorrect solution to a reasoning puzzle (similar to the problem in the textgrad paper)

**Problem**: If it takes 1 hour to dry 25 shirts under the sun, how long to dry 30 shirts?

**Wrong assumption**: More items = more time (linear scaling)

In [62]:
# The incorrect initial solution
initial_solution = """
Since 25 shirts take 1 hour, I can set up a proportion:
25 shirts : 1 hour = 30 shirts : x hours

Cross-multiplying:
25x = 30 × 1
x = 30/25 = 1.2

Therefore, it will take 1.2 hours to dry 30 shirts.
"""

print(" Initial (Incorrect) Solution:")
print(initial_solution)

 Initial (Incorrect) Solution:

Since 25 shirts take 1 hour, I can set up a proportion:
25 shirts : 1 hour = 30 shirts : x hours

Cross-multiplying:
25x = 30 × 1
x = 30/25 = 1.2

Therefore, it will take 1.2 hours to dry 30 shirts.



In [53]:
# Create a Variable - this is what we'll optimize!
solution = tg.Variable(
    initial_solution,
    requires_grad=True,  # Mark for optimization
    role_description="solution to the shirt drying problem"
)

print("Created Variable")
print(f"   requires_grad: {solution.requires_grad}")
print(f"   role: {solution.role_description}")

Created Variable
   requires_grad: True
   role: solution to the shirt drying problem


In [54]:
# Define how to evaluate the solution
evaluation_instruction = """
Evaluate this solution to the shirt drying problem.

Critical thinking required:
- Does drying time really depend on the NUMBER of items?
- All items are under the sun SIMULTANEOUSLY
- What's the fundamental flaw in the reasoning?

Point out errors clearly and constructively.
"""

loss_fn = tg.TextLoss(evaluation_instruction)
print(" Created TextLoss function")

 Created TextLoss function


In [55]:
# Create the optimizer
optimizer = tg.TGD(parameters=[solution])
print("Created TGD (Textual Gradient Descent) optimizer")

Created TGD (Textual Gradient Descent) optimizer


### Forward Pass: Evaluate the Solution

In [56]:
print("Computing loss (evaluating solution)...\n")

loss = loss_fn(solution)

print("LOSS (LLM's Critique):")
print("=" * 80)
print(loss.value)
print("=" * 80)

INFO:textgrad:LLMCall function forward


Computing loss (evaluating solution)...

LOSS (LLM's Critique):
This solution contains a fundamental logical error.

The key flaw in the reasoning:
- The solution assumes that drying time increases proportionally with the number of shirts. This is incorrect.
- In reality, when shirts are hung out to dry in the sun, they all dry simultaneously, regardless of quantity.
- If 25 shirts take 1 hour to dry under certain conditions (sun, temperature, humidity), then 30 shirts under the same conditions would also take 1 hour.

Think of it this way:
- The sun shines on all shirts at the same time
- Each shirt dries independently of the others
- Adding more shirts doesn't affect how long each individual shirt takes to dry
- The only limitation would be having enough space to hang all shirts in direct sunlight

Correct answer: The 30 shirts would still take 1 hour to dry, assuming there is sufficient space to hang them all in similar sunny conditions.

This is similar to how long it takes to cook

### Backward Pass: Get Improvement Suggestions

In [57]:
print("Computing gradients (getting suggestions)...\n")

loss.backward()

print("GRADIENTS (Improvement Suggestions):")
print("=" * 80)
print(solution.gradients)
print("=" * 80)

INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient


Computing gradients (getting suggestions)...

GRADIENTS (Improvement Suggestions):
{Variable(value=Based on the evaluation, here is key feedback for improving this solution:

1. The fundamental misconception stems from treating shirt drying as a sequential rather than parallel process. The solution needs to recognize that sun exposure happens simultaneously for all items, not one after another.

2. The proportional reasoning approach (using cross multiplication) is inappropriate here because it assumes a direct linear relationship between quantity and time. This mathematical model should be completely abandoned as it doesn't reflect the physical reality of the drying process.

3. The solution could be improved by considering the actual physical constraints that would affect drying time, such as:
   - Available space for hanging shirts in direct sunlight
   - Environmental conditions (sun intensity, temperature, humidity)
   - The fact that all shirts receive solar energy concurrently



### Optimization Step: Update the Solution

In [58]:
print("Updating solution based on gradients...\n")

optimizer.step()

print("OPTIMIZED SOLUTION:")
print("=" * 80)
print(solution.value)
print("=" * 80)

INFO:textgrad:TextualGradientDescent prompt for update
INFO:textgrad:TextualGradientDescent optimizer response
INFO:textgrad:TextualGradientDescent updated text


Updating solution based on gradients...

OPTIMIZED SOLUTION:
The drying time for 30 shirts would be the same as for 25 shirts - 1 hour. This is because clothes drying is a parallel process where all items are exposed to sunlight simultaneously, not sequentially. Just like multiple dishes can cook at the same time in an oven, multiple shirts dry at the same time in the sun.

The only requirements are:
- Sufficient space to hang all 30 shirts in direct sunlight
- Similar environmental conditions (sun intensity, temperature, humidity)

The number of shirts does not affect the drying time because each shirt dries independently and concurrently with all others. Therefore, as long as we can provide similar exposure conditions for all 30 shirts, they will dry in 1 hour.


---

# Example 2: Single-Turn Prompt Optimization

Now we optimize a system prompt to improve an LLM's performance

**Task**: Count fruits accurately

We use a cheaper model for the task, powerful model for feedback

In [59]:
# The question
question_text = """
I have a basket with:
- 3 apples
- 2 oranges
- 5 bananas
- 1 pear
- 4 grapes

How many fruits do I have in total?
"""

question = tg.Variable(
    question_text,
    requires_grad=False,  # Not optimizing the question
    role_description="counting question"
)

ground_truth = tg.Variable(
    "15",
    requires_grad=False,
    role_description="correct answer"
)

print("Question:", question_text)
print(f"\n Ground Truth: {ground_truth.value}")

Question: 
I have a basket with:
- 3 apples
- 2 oranges  
- 5 bananas
- 1 pear
- 4 grapes

How many fruits do I have in total?


 Ground Truth: 15


In [36]:
# Initial system prompt (very naive)
initial_prompt = "You are a helpful assistant."

system_prompt = tg.Variable(
    initial_prompt,
    requires_grad=True,  # optimizing this
    role_description="system prompt for accurate counting"
)

print(f"Initial System Prompt: '{initial_prompt}'")

Initial System Prompt: 'You are a helpful assistant.'


### Optimization Strategy

- **Forward engine** (task model): GPT-3.5 Turbo (cheap and fast)
- **Backward engine** (feedback): Claude 3.5 Sonnet (powerful, set earlier)

This gives us: Low-cost iterations and High-quality feedback

In [37]:
# Create model with a cheaper forward engine
forward_engine = tg.get_engine("experimental:openrouter/openai/gpt-3.5-turbo")
model = tg.BlackboxLLM(forward_engine, system_prompt=system_prompt)

print("Created model")
print("Forward engine: OpenAI GPT-3.5 Turbo (via OpenRouter)")
print("System prompt: optimizable parameter")

Created model
Forward engine: OpenAI GPT-3.5 Turbo (via OpenRouter)
System prompt: optimizable parameter


In [38]:
# Get an initial prediction
print("Getting an initial prediction...\n")

initial_prediction = model(question)

print("Initial Answer:")
print("=" * 80)
print(initial_prediction.value)
print("=" * 80)

Getting an initial prediction...



INFO:textgrad:LLMCall function forward


Initial Answer:
You have a total of 15 fruits in your basket. If you add up the quantities, it would be:

3 apples + 2 oranges + 5 bananas + 1 pear + 4 grapes = 15 fruits.


In [39]:
# Define evaluation
evaluation_prompt = f"""
The correct answer is {ground_truth.value}.

Evaluate if the prediction is correct.
If incorrect, explain:
1. What went wrong
2. How the system prompt should guide the model to:
   - Count each item individually
   - Show the calculation step-by-step
   - Verify the final sum
"""

loss_fn = tg.TextLoss(evaluation_prompt)
optimizer = tg.TGD(parameters=[system_prompt])

print(" Evaluation configured")

 Evaluation configured


In [40]:
# Run optimization
print("Running optimization...\n")

# Forward
loss = loss_fn(initial_prediction)
print("EVALUATION:")
print("=" * 80)
print(loss.value)
print("=" * 80)

# Backward
print("\n🔙 Computing gradients...\n")
loss.backward()
print("FEEDBACK ON SYSTEM PROMPT:")
print("=" * 80)
print(system_prompt.gradients)
print("=" * 80)

# Step
print("\n⚡ Updating prompt...\n")
optimizer.step()
print("OPTIMIZED SYSTEM PROMPT:")
print("=" * 80)
print(system_prompt.value)
print("=" * 80)

Running optimization...



INFO:textgrad:LLMCall function forward
INFO:textgrad:_backward_through_llm prompt


EVALUATION:
The prediction is CORRECT.

The model correctly identified that 3 + 2 + 5 + 1 + 4 = 15 fruits total.

However, the system prompt could be improved to encourage more explicit reasoning:

Suggested system prompt improvements:

1. "For any counting problem:
   - List each item and quantity separately
   - Show the addition step by step
   - Double check the final sum"

2. Example format:
   "Items:
   - Apples: 3
   - Oranges: 2  
   - Bananas: 5
   - Pear: 1
   - Grapes: 4

   Calculation:
   3 + 2 = 5
   5 + 5 = 10
   10 + 1 = 11
   11 + 4 = 15

   Verification:
   3 + 2 + 5 + 1 + 4 = 15
   Total: 15 fruits"

This structured approach would:
- Make counting more systematic
- Show clear work
- Reduce chances of errors
- Allow easy verification

🔙 Computing gradients...



INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:TextualGradientDescent prompt for update


FEEDBACK ON SYSTEM PROMPT:
{Variable(value=Since the evaluation shows that the language model already performed perfectly in this case (producing a correct count, showing work, and maintaining clarity), there isn't significant critical feedback needed for the system prompt. The model successfully:
1. Identified all quantities correctly
2. Performed accurate addition
3. Showed its work with a clear mathematical expression
4. Provided the correct total

Even though the simple "You are a helpful assistant" prompt worked well in this case, it's worth noting that this success might have been circumstantial, and the prompt could potentially fail in more complex counting scenarios. However, since there's no evidence of failure in the current objective function, I should not propose changes to fix problems that don't exist.

If future use cases present different types of counting challenges or if the evaluation metric changes, then we might need to revisit the system prompt. But for the curren

INFO:textgrad:TextualGradientDescent optimizer response
INFO:textgrad:TextualGradientDescent updated text


OPTIMIZED SYSTEM PROMPT:
You are a helpful assistant specialized in accurate counting and arithmetic. When presented with lists or sets of items to count, you will:
1. Carefully track each quantity
2. Show your mathematical work step by step
3. Double-check your calculations
4. Present the final total clearly
Always maintain precision and accuracy in your counting tasks.


In [41]:
# Test the optimized prompt
print(" Testing optimized prompt...\n")

new_prediction = model(question)

print("New Answer:")
print("=" * 80)
print(new_prediction.value)
print("=" * 80)

 Testing optimized prompt...



INFO:textgrad:LLMCall function forward


New Answer:
Let's count each type of fruit in the basket:
1. Apples: 3
2. Oranges: 2
3. Bananas: 5
4. Pear: 1
5. Grapes: 4

Now, let's add them all together:
3 apples + 2 oranges + 5 bananas + 1 pear + 4 grapes = 15 fruits

You have a total of 15 fruits in the basket.


---

# Example 3: Multi-Turn: for a Socratic Tutoring use case

This example demonstrates **real multi-turn optimization** where the idea is to tutor a "student" LLM using the Socratic method. Optimization continues until an objective is met.
-The tutor asks guiding questions instead of giving answers and the student learns through progressive refinement

Problem under consideration -  A classic reasoning puzzle

Objective: Keep tutoring until the student gets it right or max 10 iterations

In [42]:
import re

# The problem (has a common wrong answer!)
problem = """
A snail is at the bottom of a 30-foot well. Each day it climbs up 3 feet,
but each night it slips back 2 feet. How many days will it take the snail
to reach the top of the well?
"""

correct_answer = "28 days"

print(" THE PROBLEM:")
print(problem)
print(f"\n CORRECT ANSWER: {correct_answer}")
print("\n Why this is tricky:")
print("   Many think: 3-2=1 foot/day, so 30 days")
print("   But: On day 28, the snail reaches 30 feet and doesn't slip back!")

📚 THE PROBLEM:

A snail is at the bottom of a 30-foot well. Each day it climbs up 3 feet,
but each night it slips back 2 feet. How many days will it take the snail
to reach the top of the well?


 CORRECT ANSWER: 28 days

 Why this is tricky:
   Many think: 3-2=1 foot/day, so 30 days
   But: On day 28, the snail reaches 30 feet and doesn't slip back!


In [43]:
# The "student" LLM starts with the common wrong answer
initial_student_solution = """
The snail climbs 3 feet up each day but slips 2 feet back each night.
So the net progress is 3 - 2 = 1 foot per day.
To climb 30 feet at 1 foot per day, it will take 30 days.

Answer: 30 days
"""

print("STUDENT'S INITIAL SOLUTION (Wrong):")
print("=" * 80)
print(initial_student_solution)
print("=" * 80)

# Create the student's solution as an optimizable variable
student_solution = tg.Variable(
    initial_student_solution,
    requires_grad=True,
    role_description="student's evolving solution to the snail problem"
)

print("\n Student solution created as TextGrad Variable")

STUDENT'S INITIAL SOLUTION (Wrong):

The snail climbs 3 feet up each day but slips 2 feet back each night.
So the net progress is 3 - 2 = 1 foot per day.
To climb 30 feet at 1 foot per day, it will take 30 days.

Answer: 30 days


 Student solution created as TextGrad Variable


In [44]:
# Socratic tutor system - the idea is to guide the "student" without giving away the answer
socratic_instruction = f"""
You are a Socratic tutor. The student is solving:
{problem}

The correct answer is: {correct_answer}

Your role:
1. Identify errors in the student's reasoning
2. Ask guiding questions (DON'T give the answer directly!)
3. Provide hints that help them discover their mistake
4. Encourage step-by-step thinking

If the solution is correct, say: "CORRECT - The student has solved it!"
If incorrect, provide Socratic guidance to help them improve.

Be encouraging and educational. Help them learn!
"""

loss_fn = tg.TextLoss(socratic_instruction)
optimizer = tg.TGD(parameters=[student_solution])

print(" Socratic tutor configured")
print("   Role: Guide student to discover the answer")
print("   Method: Ask questions, give hints, don't tell directly")

 Socratic tutor configured
   Role: Guide student to discover the answer
   Method: Ask questions, give hints, don't tell directly


In [45]:
# Helper function to check if student got it right
def is_correct_answer(solution_text, correct_answer):
    """
    Check if the solution contains the correct answer.
    Looks for the correct number (28) mentioned as the answer.
    """
    correct_number = re.search(r'\d+', correct_answer).group()
    answer_context = solution_text.lower()

    # Check if correct number appears as the answer
    if f"{correct_number} day" in answer_context or f"answer: {correct_number}" in answer_context:
        return True
    return False

print(" Evaluation helper ready")

 Evaluation helper ready


### Tutoring Loop

The tutor guides the student through multiple iterations

-  Continues until an objective is met (not fixed iterations)
- Has a clear success criterion

In [46]:
# Multi-turn tutoring loop - continues until student succeeds!
MAX_ITERATIONS = 10
iteration = 0
student_succeeded = False

print("🎓 STARTING SOCRATIC TUTORING SESSION")
print("=" * 80)
print(f"Goal: Guide student to correct answer")
print(f"Max iterations: {MAX_ITERATIONS}")
print(f"Method: Socratic questioning\n")

while iteration < MAX_ITERATIONS:
    iteration += 1

    print("\n" + "=" * 80)
    print(f" ITERATION {iteration}")
    print("=" * 80)

    # Show current student solution
    print(f"\n Student's Current Answer:")
    print("-" * 80)
    print(student_solution.value)
    print("-" * 80)

    # Check if student has correct answer
    if is_correct_answer(student_solution.value, correct_answer):
        print("\n SUCCESS! Student reached the correct answer!")
        student_succeeded = True
        break

    # Tutor evaluates with Socratic guidance
    print(f"\n Tutor evaluating and providing guidance...\n")
    loss = loss_fn(student_solution)

    print(" Tutor's Socratic Questions & Hints:")
    print("-" * 80)
    print(loss.value)
    print("-" * 80)

    # Check if tutor confirms correctness
    if "CORRECT" in loss.value.upper() and "solved it" in loss.value.lower():
        print("\n Tutor confirms: Student has solved it!")
        student_succeeded = True
        break

    # Student reflects on tutor's questions
    print(f"\n Student processing feedback...\n")
    loss.backward()

    print("💡 Key Points to Consider:")
    print("-" * 80)
    print(student_solution.gradients)
    print("-" * 80)

    # Student revises solution
    print(f"\n✏️ Student revising answer...\n")
    optimizer.step()

    print("📄 Revised Solution:")
    print("-" * 80)
    print(student_solution.value)
    print("-" * 80)

INFO:textgrad:LLMCall function forward
INFO:textgrad:_backward_through_llm prompt
INFO:textgrad:_backward_through_llm gradient
INFO:textgrad:TextualGradientDescent prompt for update
INFO:textgrad:TextualGradientDescent optimizer response
INFO:textgrad:TextualGradientDescent updated text


🎓 STARTING SOCRATIC TUTORING SESSION
Goal: Guide student to correct answer
Max iterations: 10
Method: Socratic questioning


 ITERATION 1

 Student's Current Answer:
--------------------------------------------------------------------------------

The snail climbs 3 feet up each day but slips 2 feet back each night.
So the net progress is 3 - 2 = 1 foot per day.
To climb 30 feet at 1 foot per day, it will take 30 days.

Answer: 30 days

--------------------------------------------------------------------------------

🧑 Tutor evaluating and providing guidance...

 Tutor's Socratic Questions & Hints:
--------------------------------------------------------------------------------
Let me help guide your thinking with some questions:

1. Your calculation of net progress (3-2 = 1 foot per day) is correct. However, let's think about the final day:
- What happens when the snail finally reaches the top (30 feet)?
- Does it need to slip back that night?

2. Consider this scenario:
- At the star

In [47]:
# Show final results
print("\n\n" + "=" * 80)
print(" TUTORING SESSION RESULTS")
print("=" * 80)

if student_succeeded:
    print(f"\n SUCCESS!")
    print(f"   Student reached correct answer in {iteration} iteration(s)")
    print(f"\n Learning Progress:")
    print(f"   Initial: WRONG (30 days)")
    print(f"   Final: CORRECT ({correct_answer})")
    print(f"   Iterations: {iteration}")
else:
    print(f"\n Reached max iterations ({MAX_ITERATIONS})")
    print(f"   Student may need more guidance")

print("\n" + "=" * 80)
print("FINAL STUDENT SOLUTION")
print("=" * 80)
print(student_solution.value)
print("=" * 80)



 TUTORING SESSION RESULTS

 SUCCESS!
   Student reached correct answer in 2 iteration(s)

 Learning Progress:
   Initial: WRONG (30 days)
   Final: CORRECT (28 days)
   Iterations: 2

FINAL STUDENT SOLUTION
Let me solve this step by step:

1. Starting position: 0 feet
2. Each day:
   - Climbs up: +3 feet
   - Slips back: -2 feet
   - Net progress: 1 foot per day

3. After 27 days:
   - Net progress = 27 × 1 foot = 27 feet

4. On day 28:
   - Morning position: 27 feet
   - Climbs +3 feet = 30 feet
   - Reaches top! No need to slip back

Therefore, it will take 28 days to reach the top.

Answer: 28 days
